# Ablação H2 — Critério de Early Stopping (FT-CUR + SAINT)

Testa a hipótese H2 do relatório: o uso de `val_acc` como critério de early
stopping pode causar o colapso do FT-CUR e SAINT em datasets imbalanceados,
porque favorece checkpoints que acertam só a classe majoritária.

**Experimento:**
- Modelos: FT-CUR (Nyströmformer) e SAINT
- Datasets: CREDIT (22% pos), BANK (11% pos)
- Métricas de early stopping comparadas: `val_acc`, `val_loss`, `val_f1_macro`
- 30 seeds, total 360 runs
- Hiperparâmetros tunados reusados (varia APENAS early_stop_metric)

**Tempo esperado:**
- T4 GPU: ~1-2h
- A100: ~30-45min

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Célula 1: GPU check ─────────────────────────────────────────────────────
!nvidia-smi -L
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')

In [ ]:
# ── Célula 2: Clonar do GitHub ──────────────────────────────────────────────
import os
PROJECT_DIR = '/content/sparse-lssvm-transformers-study'
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull --rebase
else:
    !git clone {GIT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!git log --oneline -3

In [ ]:
# ── Célula 3: Dependências ──────────────────────────────────────────────────
!pip install -q optuna entmax xgboost

import numpy, scipy, sklearn, torch
print(f'numpy {numpy.__version__} | torch {torch.__version__}')

In [ ]:
# ── Célula 4: Baixar datasets ──────────────────────────────────────────────
!python scripts/download_data.py 2>&1 | tail -5

In [ ]:
# ── Célula 5: Montar Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao_tier2'
import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Drive: {DRIVE_PATH}')

In [ ]:
# ── Célula 6: Restaurar progresso anterior (resume) ─────────────────────────\n",
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_results.mkdir(exist_ok=True)
(local_results / 'tuning').mkdir(exist_ok=True)

# Resultados de experimentos
src = drive_results / 'tier2_early_stop_ablation.json'
if src.exists():
    shutil.copy(src, 'results/tier2_early_stop_ablation.json')
    print('✓ Restaurado resultado anterior dos experimentos')
else:
    print('• Começando do zero (experimentos)')

# Params tunados especificamente para cada métrica (do Drive, se existirem)
src_params = drive_results / 'tuning' / 'best_params_early_stop_ablation.json'
if src_params.exists():
    shutil.copy(src_params, 'results/tuning/best_params_early_stop_ablation.json')
    import json
    n = len(json.load(open('results/tuning/best_params_early_stop_ablation.json')))
    print(f'✓ Restaurado tuning (early_stop específico): {n} combos')
else:
    print('• Tuning específico vai começar do zero')

# Snapshot params val_acc (fallback do tier2_n5000, já vem no git)
snapshot = Path('results/tuning/best_params_tier2_n5000_snapshot.json')
if snapshot.exists():
    import json
    n = len(json.load(open(snapshot)))
    print(f'✓ Snapshot val_acc fallback: {n} combos')
else:
    print('⚠ Snapshot fallback não encontrado')

In [ ]:
# ── Célula 7: Sync para Drive em background (a cada 5 min) ──────────────────
%%writefile /content/sync_to_drive.sh
#!/bin/bash
while true; do
    sleep 300
    cp -u /content/sparse-lssvm-transformers-study/results/tier2_early_stop_ablation.json \
          "$1/tier2_early_stop_ablation.json" 2>/dev/null
    mkdir -p "$1/tuning"
    cp -u /content/sparse-lssvm-transformers-study/results/tuning/best_params_early_stop_ablation.json \
          "$1/tuning/best_params_early_stop_ablation.json" 2>/dev/null
done

In [ ]:
# ── Célula 8: Rodar ablação com TUNING específico por métrica ─────────────
# Tuning: 12 combos (2 modelos × 2 datasets × 3 métricas) × 15 trials × 3 folds
# Experimentos: 360 runs (12 combos × 30 seeds)
#
# Cada (modelo × dataset × early_stop_metric) é tunado INDEPENDENTEMENTE
# via Optuna, evitando viés de usar params tunados com val_acc para
# experimentos com val_loss/val_f1_macro.
#
# Em T4: ~3-5h (tuning ~2-3h + experimentos ~1-2h)
# Em A100: ~1.5-2h

!python scripts/run_early_stop_ablation.py --seeds 30 --trials 15 --folds 3

In [ ]:
# ── Célula 8: Rodar ablação (360 runs) ──────────────────────────────────────
# 2 modelos × 2 datasets × 3 métricas × 30 seeds = 360 runs
# Em T4: ~1-2h. Em A100: ~30-45 min.

!python scripts/run_early_stop_ablation.py --seeds 30

In [ ]:
# ── Célula 9: Salvar resultado final no Drive ───────────────────────────────
import shutil, signal
try:
    sync_proc.send_signal(signal.SIGTERM)
except Exception:
    pass

src = 'results/tier2_early_stop_ablation.json'
dst = f'{DRIVE_PATH}/tier2_early_stop_ablation.json'
shutil.copy(src, dst)

import os
size = os.path.getsize(src) / 1024
print(f'✓ Salvo no Drive: {size:.1f} KB')
!ls -lh '{DRIVE_PATH}/tier2_early_stop_ablation.json'

In [ ]:
# ── Célula 10: Análise — H2 confirmada? ─────────────────────────────────────
import json, numpy as np
from collections import defaultdict

data = json.load(open('results/tier2_early_stop_ablation.json'))
print(f'Total runs: {len(data)}  (OK: {sum(1 for r in data if r.get("status") == "ok")})\n')

scores = defaultdict(list)
for r in data:
    if r.get('status') != 'ok': continue
    key = (r['model_variant'], r['dataset'], r.get('early_stop_metric', 'val_acc'))
    scores[key].append(r.get('f1_macro', float('nan')))

print('=' * 90)
print('F1-macro: efeito do critério de early stopping')
print('=' * 90)
print(f'\n{"Modelo":<28} {"Dataset":<10} {"val_acc":>10} {"val_loss":>10} {"val_f1":>10} {"Δ vs acc":>10}')
print('─' * 90)
MODELS = ['FTTransformerCURColnorm', 'SAINTColnorm']
for m in MODELS:
    for ds in ['CREDIT', 'BANK']:
        acc = np.mean(scores.get((m, ds, 'val_acc'), [float('nan')]))
        loss = np.mean(scores.get((m, ds, 'val_loss'), [float('nan')]))
        f1m = np.mean(scores.get((m, ds, 'val_f1_macro'), [float('nan')]))
        best = max(loss, f1m) if not (np.isnan(loss) and np.isnan(f1m)) else float('nan')
        delta = best - acc if not np.isnan(best) and not np.isnan(acc) else float('nan')
        print(f'{m:<28} {ds:<10} {acc:>10.4f} {loss:>10.4f} {f1m:>10.4f} {delta:>+10.4f}')
print()
print('Interpretação:')
print('  Δ > 0   → val_loss/val_f1 melhoram. H2 (parcialmente) confirmada.')
print('  Δ < 0.02 → diferença marginal. H2 isolada não explica o colapso.')
print('  Δ ≥ 0.10 → H2 explica grande parte do colapso. H1 (balanceamento) pode não ser necessária.')